In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
B = 5
T = 8
d_k = 4
w = 4 
n_heads = 2
Q = torch.randint(0,2,(B,T,d_k),dtype = float)
K = torch.randint(0,2,(B,T,d_k),dtype = float)
V = torch.randint(0,2,(B,T,d_k),dtype = float)
mask_curr = torch.triu(torch.ones(w,w), diagonal=1).bool()
mask_prev= torch.triu(torch.ones(w,w), diagonal=1).bool()

In [45]:
h = (Q@K.transpose(-2,-1))
h = torch.unsqueeze(h, 0)
h.shape

torch.Size([1, 5, 8, 8])

In [ ]:
q_pos = torch.arange(T).view(T,1)
k_pos = torch.arange(T).view(1,T)
slopes = torch.tensor([pow(2,-8/n_heads)**i for i in range(1,n_heads+1)],dtype=float)
model = k_pos-q_pos

In [69]:
Q_chunks = [Q[:,j:j+w,:].float() for j in range(0,T,w)] #Q_chunk[0] : (B,w,d_k)
K_chunks = [K[:,j:j+w,:].float() for j in range(0,T,w)]  #K_chunks.T(-2,-1) : (B,d_k,w) 
# @ = ( B,w,d_k) @ (B,d_k,w) -> (B,w,w) @(B,w,d_k) -> (B,w,d_k)
V_chunks = [V[:,j:j+w,:].float() for j in range(0,T,w)]
chunk_curr = torch.stack([(F.softmax((Q_chunks[i]@K_chunks[i].transpose(-2,-1)/d_k**0.5).masked_fill(mask_curr, float('-inf')),dim=-1,dtype = torch.float)@V_chunks[i]) for i in range(len(Q_chunks))])

chunk_prev = torch.stack([(F.softmax((Q_chunks[i]@K_chunks[i-1].transpose(-2,-1)/d_k**0.5).masked_fill(mask_prev,float('-inf')),dim=-1,dtype = torch.float)@V_chunks[i]) for i in range(1,len(Q_chunks))])
chunk_prev[0]

tensor([[[1.0000, 1.0000, 0.0000, 1.0000],
         [0.3775, 1.0000, 0.6225, 0.3775],
         [0.6667, 1.0000, 0.6667, 0.3333],
         [0.3775, 0.7650, 0.6225, 0.1425]],

        [[0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.5000, 0.5000, 0.5000],
         [0.0000, 0.1863, 0.1863, 0.1863],
         [0.0000, 0.1888, 0.1888, 0.1888]],

        [[1.0000, 0.0000, 1.0000, 1.0000],
         [0.6225, 0.3775, 1.0000, 0.6225],
         [0.6163, 0.3837, 0.7673, 0.6163],
         [0.5000, 0.1888, 0.3775, 0.5000]],

        [[0.0000, 0.0000, 0.0000, 1.0000],
         [0.5000, 0.5000, 0.0000, 1.0000],
         [0.7673, 0.7673, 0.0000, 0.6163],
         [0.8575, 0.8575, 0.3875, 0.3775]],

        [[1.0000, 1.0000, 1.0000, 1.0000],
         [0.3775, 0.3775, 1.0000, 0.3775],
         [0.5481, 0.5481, 0.7259, 0.5481],
         [0.3276, 0.3276, 0.4599, 0.3276]]])

In [ ]:
chunk_curr = torch.stack([Q_chunks[i]@K_chunks[i].transpose(-2,-1) for i in range(len(Q_chunks))])
(_,B,w,w) = chunk_curr.shape
chunk_curr = ((chunk_curr+(model[:w,:w]*slopes[0]))/d_k**0.5).masked_fill(mask_curr, float('-inf'))
chunk_curr = F.softmax(chunk_curr,dim=-1,dtype = torch.float)
chunk_curr = torch.stack([chunk_curr[i]@V_chunks[i] for i in range(len(chunk_curr))])


chunk_prev = torch.stack([Q_chunks[i]@K_chunks[i-1].transpose(-2,-1) for i in range(1,len(Q_chunks))])
(_,B,w,w) = chunk_prev.shape
chunk_prev = ((chunk_prev+(model[:w,:w]*slopes[0]))/d_k**0.5).masked_fill(mask_prev, float('-inf'))
chunk_prev = F.softmax(chunk_prev,dim=-1,dtype = torch.float)
chunk_prev = torch.stack([chunk_prev[i-1]@V_chunks[i] for i in range(1,len(V_chunks))])
chunk_prev[0]

tensor([[[1.0000, 1.0000, 0.0000, 1.0000],
         [0.3702, 1.0000, 0.6298, 0.3702],
         [0.6668, 1.0000, 0.6770, 0.3230],
         [0.3738, 0.7543, 0.6186, 0.1357]],

        [[0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.5078, 0.5078, 0.5078],
         [0.0000, 0.1874, 0.1874, 0.1874],
         [0.0000, 0.1864, 0.1864, 0.1864]],

        [[1.0000, 0.0000, 1.0000, 1.0000],
         [0.6151, 0.3849, 1.0000, 0.6151],
         [0.6146, 0.3854, 0.7589, 0.6146],
         [0.4922, 0.1843, 0.3630, 0.4922]],

        [[0.0000, 0.0000, 0.0000, 1.0000],
         [0.5078, 0.5078, 0.0000, 1.0000],
         [0.7756, 0.7756, 0.0000, 0.6061],
         [0.8656, 0.8656, 0.4012, 0.3630]],

        [[1.0000, 1.0000, 1.0000, 1.0000],
         [0.3702, 0.3702, 1.0000, 0.3702],
         [0.5483, 0.5483, 0.7173, 0.5483],
         [0.3224, 0.3224, 0.4452, 0.3224]]])

In [ ]:
q_pos = torch.arange(T).view(T,1)
k_pos = torch.arange(T).view(1,T)
k_pos-q_pos
x = chunk_curr



ValueError: not enough values to unpack (expected 4, got 3)

In [ ]:
n_heads = 8
slopes = torch.tensor([pow(2,-8/n_heads)**i for i in range(1,n_heads+1)],dtype=float)

In [212]:
k = T-1
W = torch.randn((2*k+1,d_k),dtype=float)
q_pos = torch.arange(T).view(T,1)
k_pos = torch.arange(T).view(1,T)
relative = (q_pos-k_pos).clamp(-k,k)+k #The T*T matrix
#assign an embedding to each i_j : 2k+1 embedding 
relative.shape

torch.Size([8, 8])

In [213]:
embed = torch.nn.Embedding(2*k+1,d_k)
R = embed(relative)
R.shape

torch.Size([8, 8, 4])

In [ ]:
res = torch.einsum("bid,ijd->bij",Q.float(),R.float())

tensor([[[ 7.0113e-02,  1.9775e+00, -4.1777e-01,  3.6332e-01,  6.1219e-01,
          -3.5928e-01,  1.4334e+00,  7.0035e-01],
         [ 8.4055e-01, -4.7064e-01,  1.3241e+00,  5.4920e-01,  2.1788e+00,
           1.5153e+00, -2.2613e+00, -1.7116e+00],
         [ 6.4931e-01,  5.5775e-01, -4.0053e-01,  3.3016e+00,  1.3143e-01,
           2.5421e+00,  2.1275e+00, -2.6206e+00],
         [-7.0020e-01, -1.0686e-01,  2.8678e-01, -1.1435e+00,  3.7252e-01,
           1.5142e+00,  9.9312e-01,  9.2045e-01],
         [ 6.1769e-01, -1.9859e+00, -2.2814e+00,  7.6825e-01, -4.4701e-01,
           1.3757e+00, -7.7759e-01,  1.7274e+00],
         [-9.0873e-02,  1.3979e+00, -1.2995e+00, -1.5561e-01,  7.7222e-01,
          -1.5204e+00,  3.7257e+00,  3.1884e-01],
         [ 1.8396e-02,  3.4095e-01,  1.1593e+00, -9.8054e-01, -8.0492e-01,
           2.1448e-01, -1.1199e+00,  4.2408e-01],
         [-1.3642e+00,  1.5695e+00,  4.6921e-01,  2.3589e-01, -1.7056e+00,
          -1.5833e+00,  8.4055e-01, -4.7064e-01]],